# 04. ネスト構造・判別共用体（Discriminated Union）・JSON Schema

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/ja/04_nested_unions_and_schema.ipynb)

実務の REST API、Webhook イベントストリーム、および LLM のツール呼び出し／構造化出力では、単一のフラットなモデルだけで完結することは稀です。本番システムでは **ネストされた `BaseModel` 階層** を組み立て、**タグ付き判別共用体（`Field(discriminator=...)` + `TypeAdapter`）** によって多態的なペイロードを `O(1)` で高速振り分けし、さらに **`model_json_schema()`** から出力される標準 JSON Schema を FastAPI の OpenAPI 3.1 ドキュメント生成や LLM の文法制約付きデコーディング（Google GenAI SDK の `response_schema` など）で一元活用します。

![図4-1: タグ付き判別共用体（Discriminated Union）と TypeAdapter による高速振り分け](https://examples.drawlib.com/pydantic/images/04_nested_unions_and_schema/ch04_discriminated_union_ja.png)

*図4-1: タグ付き判別共用体（Discriminated Union）と TypeAdapter による高速振り分け*

---

## 1. ネスト構造と判別共用体（`Field(discriminator=...)` + `TypeAdapter`）

通常のタグなし共用体（`CardPayment | BankTransfer | WalletPayment`）では、バリデータが左から順番にすべての候補モデルを総当たりで試行するため、不一致時の CPU コストが増大するだけでなく、入力不備の際に全候補分の冗長な `ValidationError` が発生してしまいます。各サブモデルに `Literal[...]` 型のタグフィールド（`kind` など）を持たせ、`Annotated[..., Field(discriminator="kind")]` で束ねると、`pydantic-core` 内部での分岐が **`O(1)` のハッシュテーブル検索** へと最適化されます。

- **ネストされた `BaseModel` 合成**: `CheckoutOrder` の属性として `PaymentMethod` を埋め込むだけで、ネストされた辞書が1回の検証パスで適切な具象クラス（`CardPayment` / `BankTransfer` / `WalletPayment`）へと変換されます。
- **`TypeAdapter` によるコレクションの直接検証**: Webhook のイベント配列など、外側のラッパー `BaseModel` を持たない `list[PaymentMethod]` を直接検証したい場合は、モジュール読み込み時に `TypeAdapter(list[PaymentMethod])` を1回コンパイルして再利用します。
- **ピンポイントなエラー特定**: `{"kind": "card", "last4": "12AB"}` のように正規表現制約に違反した場合でも、`CardPayment` へ直行するため、`bank_code` や `provider` の欠落ノイズを含まない `'card' -> 'last4'` だけの正確なエラーが得られます。

In [ ]:
!pip install -q pydantic

In [ ]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field, TypeAdapter, ValidationError


class CardPayment(BaseModel):
    kind: Literal["card"] = "card"
    last4: str = Field(pattern=r"^\d{4}$", description="マスク済みのカード番号下4桁。")
    amount: int = Field(gt=0, description="決済金額（セント単位）。")


class BankTransfer(BaseModel):
    kind: Literal["bank"] = "bank"
    bank_code: str = Field(min_length=4, max_length=11, description="SWIFTコードまたは銀行識別コード。")
    amount: int = Field(gt=0, description="振込金額（セント単位）。")


class WalletPayment(BaseModel):
    kind: Literal["wallet"] = "wallet"
    provider: Literal["apple_pay", "google_pay"] = Field(description="デジタルウォレット種別。")
    amount: int = Field(gt=0, description="決済金額（セント単位）。")


# 'kind' リテラル値で O(1) ルーティングを行う判別共用体（Discriminated Union）
PaymentMethod = Annotated[
    CardPayment | BankTransfer | WalletPayment,
    Field(discriminator="kind"),
]


class CheckoutOrder(BaseModel):
    order_id: str = Field(description="一意の注文ID。")
    customer_email: str = Field(description="購入者の確認用メールアドレス。")
    payment: PaymentMethod = Field(description="決済手段ごとの多態ペイロード。")


# 1. ネストされた CheckoutOrder ペイロードの検証
order = CheckoutOrder.model_validate(
    {
        "order_id": "ORD-2026-0042",
        "customer_email": "alice@example.com",
        "payment": {"kind": "card", "last4": "4242", "amount": 4800},
    }
)
print(f"Nested order {order.order_id} -> {type(order.payment).__name__}: {order.payment}")

# 2. TypeAdapter を用いた list[PaymentMethod] バッチ配列の直接検証
payment_batch_adapter = TypeAdapter(list[PaymentMethod])
payments = payment_batch_adapter.validate_python(
    [
        {"kind": "card", "last4": "4242", "amount": 4800},
        {"kind": "bank", "bank_code": "BOFAUS3N", "amount": 12500},
        {"kind": "wallet", "provider": "google_pay", "amount": 1900},
    ]
)
print("Batch routed types:", [type(item).__name__ for item in payments])

# 3. 特定バリアントのフィールド不正時におけるピンポイントな O(1) エラー報告
try:
    payment_batch_adapter.validate_python(
        [{"kind": "card", "last4": "12AB", "amount": 4800}]
    )
except ValidationError as exc:
    err = exc.errors(include_url=False)[0]
    print(f"Pinpoint error ({exc.error_count()} total): loc={err['loc']} -> {err['msg']}")

---

## 2. API・LLM 連携のための標準 JSON Schema 生成（`model_json_schema()`）

実行時の高速バリデーションに加えて、Pydantic v2 のすべての `BaseModel`（および `TypeAdapter`）は `.model_json_schema()` メソッドによって **JSON Schema Draft 2020-12** 準拠の辞書を直接出力できます。型制約・`Literal` 分類・`Field(discriminator=...)` マッピング・`Field(description=...)` の説明文を Python コード上で1回定義するだけで、システム全体の **Single Source of Truth（唯一の情報源）** として機能します。

![図4-2: 単一の Pydantic モデルから FastAPI (OpenAPI) と LLM 構造化出力へのスキーマ展開](https://examples.drawlib.com/pydantic/images/04_nested_unions_and_schema/ch04_json_schema_ecosystem_ja.png)

*図4-2: 単一の Pydantic モデルから FastAPI (OpenAPI) と LLM 構造化出力へのスキーマ展開*

`CheckoutOrder.model_json_schema()` を呼び出すと、再利用可能なサブモデル定義（`$defs`）、判別キーと参照先の対応表（`discriminator`）、必須属性リスト（`required`）、正規表現や数値範囲の制約、そして各フィールドの `description` メタデータが一括出力されます（日本語の `description` を出力するため `ensure_ascii=False` を指定しています）。次のショーケースである **Google GenAI SDK Cookbook** で `response_schema=CheckoutOrder` を渡した際も、内部ではまさにこの JSON Schema 生成機構が呼び出され、Gemini のトークン生成が有効な `CheckoutOrder` 形式へと厳密に制約されます。

In [ ]:
import json

schema = CheckoutOrder.model_json_schema()

print("Top-level schema keys:", list(schema.keys()))
print("Nested $defs models:", list(schema["$defs"].keys()))
print("\nFull JSON Schema (Draft 2020-12):")
print(json.dumps(schema, indent=2, ensure_ascii=False))